# 01 · Download en data-audit

**Auteurs:** _vul namen en concrete bijdragen in vóór indiening_  
**Doel:** Officiële maandarchieven met code downloaden, valideren en de herkomst vastleggen.

Elke codecel wordt voorafgegaan door uitleg. Voer de notebooks in nummervolgorde uit
vanaf de repository-root. Resultaten moeten door het team zelf worden gecontroleerd en
geïnterpreteerd; synthetische CI-data gelden nooit als onderzoeksbewijs.

In [ ]:
# Maak lokale projectimports betrouwbaar in VS Code, JupyterLab en nbconvert.
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Projectroot niet gevonden; open de map Brent als VS Code-workspace.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Projectroot: {PROJECT_ROOT}")

## Keuze van periode

De standaardrun gebruikt het volledige kalenderjaar 2024, zodat alle seizoenen in de
analyse voorkomen. De periode is centraal vastgelegd, zodat scripts en notebooks
dezelfde grenzen gebruiken en cherry-picking achteraf wordt voorkomen.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from src.config import DEFAULT_END_MONTH, DEFAULT_START_MONTH, RAW_DIR, WEATHER_PATH
from src.data.download import download_range
from src.data.weather import download_hourly_weather

START_MONTH = DEFAULT_START_MONTH
END_MONTH = DEFAULT_END_MONTH
RUN_DOWNLOAD = False    # zet bewust op True; download kan meerdere GB zijn
if RUN_DOWNLOAD:
    download_range(START_MONTH, END_MONTH)
    download_hourly_weather(START_MONTH, END_MONTH, WEATHER_PATH)

## Controle van provenance

Het manifest maakt elke bronfile controleerbaar met URL, grootte en SHA-256.

In [ ]:
manifest_path = RAW_DIR / "manifest.json"
if not manifest_path.exists():
    print("Nog geen manifest. Zet RUN_DOWNLOAD=True en voer de vorige cel uit.")
else:
    manifest = pd.DataFrame(json.loads(manifest_path.read_text()))
    display(manifest)
    print(f"Totaal: {manifest['bytes'].sum() / 1e9:.2f} GB")
    assert manifest["sha256"].str.len().eq(64).all()

## Schema-audit zonder alles in geheugen te laden

We lezen alleen de headers en vergelijken kolommen per CSV-lid.

In [ ]:
from zipfile import ZipFile

schemas = []
for path in sorted(RAW_DIR.glob("*-citibike-tripdata.zip")):
    with ZipFile(path) as archive:
        for member in archive.namelist():
            if member.lower().endswith(".csv"):
                with archive.open(member) as handle:
                    columns = pd.read_csv(handle, nrows=0).columns.tolist()
                schemas.append({"archive": path.name, "member": member, "columns": columns})
display(pd.DataFrame(schemas))

## Besluit

Het downloadscript faalt expliciet bij corrupte ZIP's en schrijft pas na succesvolle
validatie. Schema-afwijkingen worden hier zichtbaar voordat cleaning start. Noteer in
de definitieve versie de gedownloade periode, omvang en eventuele afwijkingen.